# Notebook for experiments

In [ ]:
import abc
import json
import logging
import os
import re
import signal
import sys
import time
from typing import Optional, List, Dict, Any
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import wikipediaapi
import flickrapi

In [26]:
# Configure logging with timestamps and log levels.
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Global flag for graceful stop
STOP_FLAG = False

def signal_handler(sig, frame):
    global STOP_FLAG
    logging.info("Received stop signal. Preparing to exit after current checkpoint...")
    STOP_FLAG = True

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

<function __main__.signal_handler(sig, frame)>

In [ ]:
# ----- Define Full List of Places in Russia and Keywords -----
PLACES = [
    "Москва", "Санкт-Петербург", "Казань", "Сочи", "Екатеринбург",
    "Новосибирск", "Владивосток", "Нижний Новгород", "Самара", "Волгоград",
    "Ростов-на-Дону", "Калининград", "Иркутск", "Суздаль", "Владимир",
    "Архангельск", "Мурманск", "Пермь", "Омск", "Уфа", "Челябинск"
]

KEYWORDS = [
    "история", "архитектура", "музей", "достопримечательность", "культура",
    "памятник", "церковь", "собор", "парк", "природа", "фестиваль", "кухня",
    "традиция", "экскурсия", "легенда", "знаменит", "археология", "туризм"
]

In [28]:
# ----- Directory Structure for Checkpoints and Aggregated Data -----
BASE_CHECKPOINT_DIR = "./checkpoints"
RESOURCES = ["wikipedia", "flickr", "scraper", "osm"]

for resource in RESOURCES:
    dir_path = os.path.join(BASE_CHECKPOINT_DIR, resource)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

AGGREGATED_DIR = "./aggregated"
if not os.path.exists(AGGREGATED_DIR):
    os.makedirs(AGGREGATED_DIR)

In [29]:
# ----- Utility Function for Filename Normalization -----
def normalize_filename(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[^\w_]', '', name)
    return name

In [30]:
# ----- Abstract Base Parser -----
class BaseParser(abc.ABC):
    @abc.abstractmethod
    def parse(self, place: str, **kwargs) -> Optional[Any]:
        pass

In [31]:
# ----- Wikipedia Parser with Checkpoint -----
class WikipediaParser(BaseParser):
    def __init__(self, user_agent: str = 'your-user-agent', language: str = 'ru'):
        self.wiki = wikipediaapi.Wikipedia(user_agent=user_agent, language=language)

    def parse(self, place: str, checkpoint: bool = True,
              checkpoint_dir: str = os.path.join(BASE_CHECKPOINT_DIR, "wikipedia"), **kwargs) -> Optional[Dict[str, str]]:
        logging.info(f"[Wikipedia] Searching page for: {place}")
        page = self.wiki.page(place)
        if page.exists():
            logging.info(f"[Wikipedia] Found page: {page.title}")
            wiki_data = {
                "title": page.title,
                "summary": page.summary,
                "text": page.text,
            }
            if checkpoint:
                cp_filename = os.path.join(checkpoint_dir, f"wikipedia_{normalize_filename(place)}.json")
                try:
                    with open(cp_filename, "w", encoding="utf-8") as f:
                        json.dump(wiki_data, f, ensure_ascii=False, indent=4)
                    logging.info(f"[Wikipedia] Checkpoint saved: {cp_filename}")
                except Exception as e:
                    logging.error(f"[Wikipedia] Error saving checkpoint: {e}")
            return wiki_data
        else:
            logging.warning(f"[Wikipedia] Page '{place}' does not exist.")
            return None

In [32]:
# ----- Flickr Parser -----
class FlickrParser(BaseParser):
    def __init__(self, api_key: str, api_secret: str, response_format: str = 'parsed-json'):
        self.flickr = flickrapi.FlickrAPI(api_key, api_secret, format=response_format)

    def parse(self, place: str, per_page: int = 100, max_pages: Optional[int] = None,
              checkpoint: bool = True, checkpoint_dir: str = os.path.join(BASE_CHECKPOINT_DIR, "flickr"), **kwargs) -> Optional[List[Dict[str, str]]]:
        tag = place  # Use place name as the search tag.
        logging.info(f"[Flickr] Searching for tag: {tag}")
        aggregated_results = []
        try:
            initial_response = self.flickr.photos.search(tags=tag, per_page=per_page, page=1)
            photos_info = initial_response.get('photos', {})
            total_pages = photos_info.get('pages', 1)
            logging.info(f"[Flickr] {total_pages} pages available for tag '{tag}'.")
            if max_pages is not None:
                total_pages = min(total_pages, max_pages)
            for page in range(1, total_pages + 1):
                if STOP_FLAG:
                    logging.info("[Flickr] Stop flag detected. Exiting Flickr parsing loop.")
                    break
                logging.info(f"[Flickr] Processing page {page} of {total_pages} for tag '{tag}'.")
                response = self.flickr.photos.search(tags=tag, per_page=per_page, page=page)
                photos = response.get('photos', {}).get('photo', [])
                for photo in photos:
                    photo_id = photo.get('id')
                    photo_title = photo.get('title')
                    owner = photo.get('owner')
                    photo_url = f"https://www.flickr.com/photos/{owner}/{photo_id}"
                    aggregated_results.append({
                        "id": photo_id,
                        "title": photo_title,
                        "url": photo_url
                    })
                # Save checkpoint after each page.
                cp_filename = os.path.join(checkpoint_dir, f"flickr_{normalize_filename(place)}_page_{page}.json")
                with open(cp_filename, "w", encoding="utf-8") as f:
                    json.dump(aggregated_results, f, ensure_ascii=False, indent=4)
                logging.info(f"[Flickr] Checkpoint saved: {cp_filename}")
            logging.info(f"[Flickr] Aggregated {len(aggregated_results)} photos for tag '{tag}'.")
            return aggregated_results
        except Exception as e:
            logging.error(f"[Flickr] Error for tag '{tag}': {e}")
            return None

In [33]:
# ----- Web Scraper Parser (например, LonelyPlanet) -----
class WebScraperParser(BaseParser):
    def __init__(self, user_agent: Optional[str] = None, timeout: int = 10):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": user_agent or (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/114.0.0.0 Safari/537.36")
        })
        self.timeout = timeout

    def parse(self, place: str, css_selector: str, next_page_selector: Optional[str] = None,
              max_pages: Optional[int] = None, checkpoint: bool = True, checkpoint_dir: str = os.path.join(BASE_CHECKPOINT_DIR, "scraper"), **kwargs) -> Optional[List[Dict[str, Optional[str]]]]:
        base_url = f"https://www.lonelyplanet.com/search?q={place}"
        logging.info(f"[Scraper] Starting scraping for '{place}' from {base_url}")
        aggregated_results = []
        current_url = base_url
        page_count = 0

        while current_url:
            if STOP_FLAG:
                logging.info("[Scraper] Stop flag detected. Exiting scraper loop.")
                break

            page_count += 1
            logging.info(f"[Scraper] Processing page {page_count}: {current_url}")
            try:
                response = self.session.get(current_url, timeout=self.timeout)
                response.raise_for_status()
            except requests.RequestException as re:
                logging.error(f"[Scraper] Error fetching {current_url}: {re}")
                break

            soup = BeautifulSoup(response.content, "html.parser")
            elements = soup.select(css_selector)
            logging.info(f"[Scraper] Found {len(elements)} elements on page {page_count}")
            for element in elements:
                text = element.get_text(strip=True)
                link = element.get("href")
                aggregated_results.append({
                    "text": text,
                    "link": link
                })
            # Save checkpoint after each page.
            cp_filename = os.path.join(checkpoint_dir, f"scraper_{normalize_filename(place)}_page_{page_count}.json")
            with open(cp_filename, "w", encoding="utf-8") as f:
                json.dump(aggregated_results, f, ensure_ascii=False, indent=4)
            logging.info(f"[Scraper] Checkpoint saved: {cp_filename}")
            if max_pages is not None and page_count >= max_pages:
                logging.info(f"[Scraper] Reached max_pages limit: {max_pages}.")
                break
            if next_page_selector:
                next_page_element = soup.select_one(next_page_selector)
                if next_page_element and next_page_element.get("href"):
                    next_href = next_page_element.get("href")
                    current_url = urljoin(current_url, next_href)
                    logging.info(f"[Scraper] Next page: {current_url}")
                else:
                    logging.info("[Scraper] No next page link found. Ending pagination.")
                    break
            else:
                break

        logging.info(f"[Scraper] Aggregated {len(aggregated_results)} items for '{place}'.")
        return aggregated_results if aggregated_results else None

In [34]:
# ----- OSM Parser using Overpass API -----
class OSMParser(BaseParser):
    """
    Uses Overpass API to retrieve place information from OpenStreetMap.
    Пример запроса: достопримечательности (tourism=attraction) в заданном месте.
    """
    def __init__(self, overpass_url: str = "https://overpass-api.de/api/interpreter", timeout: int = 25):
        self.overpass_url = overpass_url
        self.timeout = timeout

    def parse(self, place: str, radius: int = 10000, checkpoint: bool = True, checkpoint_dir: str = os.path.join(BASE_CHECKPOINT_DIR, "osm"), **kwargs) -> Optional[List[Dict[str, Any]]]:
        logging.info(f"[OSM] Querying Overpass API for place: {place}")
        query = f"""
        [out:json][timeout:{self.timeout}];
        area["name"="{place}"];
        (
          node["tourism"="attraction"](area);
          way["tourism"="attraction"](area);
          relation["tourism"="attraction"](area);
        );
        out center;
        """
        try:
            response = requests.post(self.overpass_url, data={"data": query}, timeout=self.timeout)
            response.raise_for_status()
            data = response.json()
            elements = data.get("elements", [])
            results = []
            for elem in elements:
                result = {
                    "id": elem.get("id"),
                    "type": elem.get("type"),
                    "tags": elem.get("tags", {}),
                    "lat": elem.get("lat") or elem.get("center", {}).get("lat"),
                    "lon": elem.get("lon") or elem.get("center", {}).get("lon")
                }
                results.append(result)
            # Save checkpoint for OSM data for this place.
            cp_filename = os.path.join(checkpoint_dir, f"osm_{normalize_filename(place)}.json")
            with open(cp_filename, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            logging.info(f"[OSM] Checkpoint saved: {cp_filename}")
            return results
        except Exception as e:
            logging.error(f"[OSM] Error querying Overpass API for {place}: {e}")
            return None

In [35]:
# ----- Data Processor -----
class DataProcessor:
    def __init__(self, keywords: List[str]):
        self.keywords = keywords

    def process_text(self, text: str) -> List[str]:
        cleaned_text = re.sub(r'\s+', ' ', text)
        sentences = re.split(r'[.!?]', cleaned_text)
        interesting_sentences = [
            sentence.strip() for sentence in sentences
            if any(keyword in sentence.lower() for keyword in self.keywords)
        ]
        return interesting_sentences


In [36]:
# ----- Composite Aggregator -----
class CompositeAggregator:
    def __init__(self,
                 wikipedia_parser: WikipediaParser,
                 flickr_parser: FlickrParser,
                 scraper_parser: WebScraperParser,
                 osm_parser: OSMParser,
                 data_processor: DataProcessor):
        self.wikipedia_parser = wikipedia_parser
        self.flickr_parser = flickr_parser
        self.scraper_parser = scraper_parser
        self.osm_parser = osm_parser
        self.data_processor = data_processor

    def aggregate(self, place: str, flickr_max_pages: Optional[int] = None,
                  scraper_max_pages: Optional[int] = None) -> Dict[str, Any]:
        data = {"place": place}
        agg_filename = os.path.join(AGGREGATED_DIR, f"{normalize_filename(place)}.json")
        if os.path.exists(agg_filename):
            logging.info(f"[Aggregate] Data for '{place}' already exists in {agg_filename}. Skipping aggregation.")
            try:
                with open(agg_filename, "r", encoding="utf-8") as f:
                    data = json.load(f)
                return data
            except Exception as e:
                logging.error(f"[Aggregate] Error loading existing data for '{place}': {e}")
        
        # Wikipedia data.
        wiki_data = self.wikipedia_parser.parse(place, checkpoint=True)
        if wiki_data:
            data["wikipedia"] = wiki_data
            interesting_facts = self.data_processor.process_text(wiki_data.get("summary", ""))
            data["interesting_facts"] = interesting_facts
        else:
            data["wikipedia"] = None
            data["interesting_facts"] = []
        
        # Flickr data.
        flickr_data = self.flickr_parser.parse(place, max_pages=flickr_max_pages)
        data["flickr"] = flickr_data if flickr_data is not None else []
        
        # Web Scraper (LonelyPlanet)
        css_selector = "a.card-link" 
        next_page_selector = "a.pagination__next" 
        scraper_data = self.scraper_parser.parse(place, css_selector=css_selector,
                                                 next_page_selector=next_page_selector,
                                                 max_pages=scraper_max_pages)
        data["travel_info"] = scraper_data if scraper_data is not None else []
        
        # OSM (Overpass API)
        osm_data = self.osm_parser.parse(place)
        data["osm"] = osm_data if osm_data is not None else []

        return data

In [37]:
# ----- Utility Function for Saving JSON -----
def save_json(data: Dict[str, Any], filename: str):
    try:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        logging.info(f"[Save] Data for '{data.get('place')}' saved to {filename}")
    except Exception as e:
        logging.error(f"[Save] Error saving data to {filename}: {e}")

In [ ]:
# ----- Main Execution -----
def main():
    wiki_parser = WikipediaParser(
        user_agent='Mozilla/5.0 (X11; Linux x86_64; rv:135.0) Gecko/20100101 Firefox/135.0', language='ru'
    )
    flickr_parser = FlickrParser(
        api_key='deb54a3c3e5febaf6590af51a8571a58',
        api_secret='42180b746b4d74c7'
    )
    scraper_parser = WebScraperParser()
    osm_parser = OSMParser()
    data_processor = DataProcessor(KEYWORDS)

    aggregator = CompositeAggregator(
        wikipedia_parser=wiki_parser,
        flickr_parser=flickr_parser,
        scraper_parser=scraper_parser,
        osm_parser=osm_parser,
        data_processor=data_processor
    )

    flickr_max_pages = 2
    scraper_max_pages = 2

    all_data = {}
    for place in PLACES:
        if STOP_FLAG:
            logging.info("Stop flag detected in main loop. Exiting before processing next place.")
            break
        logging.info(f"[Aggregate] Aggregating data for: {place}")
        data = aggregator.aggregate(place, flickr_max_pages=flickr_max_pages, scraper_max_pages=scraper_max_pages)
        agg_filename = os.path.join(AGGREGATED_DIR, f"{normalize_filename(place)}.json")
        save_json(data, agg_filename)
        all_data[place] = data
        time.sleep(2)
    
    # Save all aggregated data to one JSON file.
    aggregated_filename = os.path.join(AGGREGATED_DIR, "all_places_aggregated.json")
    save_json(all_data, aggregated_filename)
    logging.info(f"[Aggregate] All data aggregated and saved to {aggregated_filename}")

if __name__ == '__main__':
    main()


2025-02-12 16:02:16,655 [INFO] Wikipedia: language=ru, user_agent: Mozilla/5.0 (X11; Linux x86_64; rv:135.0) Gecko/20100101 Firefox/135.0 (Wikipedia-API/0.8.1; https://github.com/martin-majlis/Wikipedia-API/), extract_format=ExtractFormat.WIKI
2025-02-12 16:02:16,663 [INFO] [Aggregate] Aggregating data for: Москва
2025-02-12 16:02:16,665 [INFO] [Aggregate] Data for 'Москва' already exists in ./aggregated/москва.json. Skipping aggregation.


2025-02-12 16:02:16,738 [INFO] [Save] Data for 'Москва' saved to ./aggregated/москва.json
2025-02-12 16:02:18,747 [INFO] [Aggregate] Aggregating data for: Санкт-Петербург
2025-02-12 16:02:18,753 [INFO] [Aggregate] Data for 'Санкт-Петербург' already exists in ./aggregated/санктпетербург.json. Skipping aggregation.
2025-02-12 16:02:18,936 [INFO] [Save] Data for 'Санкт-Петербург' saved to ./aggregated/санктпетербург.json
2025-02-12 16:02:20,939 [INFO] [Aggregate] Aggregating data for: Казань
2025-02-12 16:02:20,940 [INFO] [Aggregate] Data for 'Казань' already exists in ./aggregated/казань.json. Skipping aggregation.
2025-02-12 16:02:20,949 [INFO] [Save] Data for 'Казань' saved to ./aggregated/казань.json
2025-02-12 16:02:22,953 [INFO] [Aggregate] Aggregating data for: Сочи
2025-02-12 16:02:22,958 [INFO] [Aggregate] Data for 'Сочи' already exists in ./aggregated/сочи.json. Skipping aggregation.
2025-02-12 16:02:23,064 [INFO] [Save] Data for 'Сочи' saved to ./aggregated/сочи.json
2025-02-12